# Golden Set v3.1 + Blind VLM 통합 실험 (L4 안정화본)

기존 검색기와 생성기는 수정하지 않습니다. B15/G22/doc_id 채점 보정과 정답 위치를 사용하지 않는 VLM 연결만 검증합니다.

커널은 `myenv`를 선택하고 Restart Kernel 후 위에서부터 실행합니다.

In [ ]:
import getpass
import json
import os
import subprocess
import time
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path('/home/kongseok/sprint-public-procurement-rag-assistant')
os.chdir(ROOT)
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # 검색용 KURE는 CPU

archive_candidates = [
    ROOT / 'denoising-dirty-documents.zip',
    ROOT / 'denoising-dirty-documents.Zip',
    Path('/home/kongseok/denoising-dirty-documents.zip'),
    Path('/home/kongseok/denoising-dirty-documents.Zip'),
]
ARCHIVE_PATH = next((path for path in archive_candidates if path.exists()), None)
assert ARCHIVE_PATH, '원본 압축 파일이 없습니다.'
assert (ROOT / 'output/chunks.pkl').exists(), 'output/chunks.pkl이 없습니다.'

if not os.environ.get('OPENAI_API_KEY', '').startswith('sk-'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API Key: ').strip()
assert os.environ['OPENAI_API_KEY'].startswith('sk-'), 'API 키를 확인하세요.'
print('준비 완료:', ARCHIVE_PATH)

In [ ]:
# Qwen3-VL 서버를 L4 24GB에 맞춘 보수적인 설정으로 한 번만 시작합니다.
VLM_URL = 'http://127.0.0.1:8003/v1'
VLM_MODEL = 'Qwen/Qwen3-VL-8B-Instruct'
vlm_process = None
vlm_log_handle = None

def server_models():
    try:
        with urllib.request.urlopen(VLM_URL + '/models', timeout=3) as response:
            return json.load(response)
    except Exception:
        return None

models = server_models()
if models is None:
    free_mib = int(subprocess.check_output([
        'nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'
    ], text=True).strip().splitlines()[0])
    if free_mib < 18000:
        subprocess.run(['nvidia-smi'])
        raise RuntimeError(f'VLM 시작 전 GPU 여유 메모리가 부족합니다: {free_mib} MiB')

    log_path = ROOT / 'output/vlm_blind_server_l4.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    vlm_log_handle = log_path.open('w', encoding='utf-8')
    shell_command = r'''
unset PYTHONPATH PYTHONHOME LD_PRELOAD
source /home/kongseok/vllm-venv/bin/activate
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
export LD_LIBRARY_PATH="$VIRTUAL_ENV/lib/python3.12/site-packages/torch/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cu13/lib"
exec "$VIRTUAL_ENV/bin/vllm" serve Qwen/Qwen3-VL-8B-Instruct --host 127.0.0.1 --port 8003 --gpu-memory-utilization 0.72 --max-model-len 4096 --max-num-seqs 1
'''
    vlm_process = subprocess.Popen(
        ['/bin/bash', '-lc', shell_command], cwd=ROOT,
        stdout=vlm_log_handle, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print('VLM 서버 시작 중:', log_path)
    for _ in range(180):
        if vlm_process.poll() is not None:
            vlm_log_handle.flush()
            print(log_path.read_text(encoding='utf-8')[-4000:])
            raise RuntimeError('VLM 서버 시작 실패')
        models = server_models()
        if models is not None:
            break
        time.sleep(5)
    else:
        raise TimeoutError('VLM 서버 준비 시간 초과')

served = [item.get('id') for item in models.get('data', [])]
assert VLM_MODEL in served, f'다른 모델 서버가 실행 중입니다: {served}'
print('VLM 서버 준비 완료:', served)

In [ ]:
# 전체 실험 실행. 완료 또는 오류 시 이 노트북이 시작한 VLM 서버를 종료합니다.
from experiments.vlm_blind_v1.run_experiment import run

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESULT_DIR = None
try:
    RESULT_DIR = run(ARCHIVE_PATH, RUN_ID)
finally:
    if vlm_process is not None and vlm_process.poll() is None:
        vlm_process.terminate()
        try:
            vlm_process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            vlm_process.kill()
    if vlm_log_handle is not None:
        vlm_log_handle.close()
print('결과 폴더:', RESULT_DIR)

In [ ]:
import pandas as pd
summary = json.loads((RESULT_DIR / 'summary.json').read_text(encoding='utf-8'))
display(pd.DataFrame([summary]).T.rename(columns={0: '결과'}))
print('정답 위치 사용:', summary['gold_document_locator_used'])
print('채점기:', summary['scorer_version'])
print('골든셋 패치:', summary['golden_patch_version'])